In [3]:
import yaml 
from typing import List
import multiprocess
from tqdm import tqdm 
import os 
from urllib.parse import urlparse
import json 
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
import jellyfish
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from typing import Sequence

prefix = "/Volumes/New Volume/malware-detection-dataset/winget-pkgs-master/manifests"
path = f"{prefix}/manifest-files.txt"

with open(path, "r") as file: 
    manifests = file.read().splitlines()
    
manifests = [os.path.join(prefix, man_path) for man_path in manifests]

In [ ]:
from typing import Callable


def parse_manifest(path: str) -> dict:
    """Parses a YAML manifest file."""
    with open(path, "r", encoding="utf-8") as file:
        return yaml.safe_load(file)

def extract_url(manifest: dict) -> List[str]:
    """Extracts installer URLs from a manifest."""
    if manifest.get("Installers") is None: 
        return [] 
    
    return [installer["InstallerUrl"] for installer in manifest["Installers"]]

def process_manifest(path: str, process_fn: Callable) -> List[str]:
    """Processes a single manifest file, returning installer URLs if found."""
    try:
        manifest = parse_manifest(path)
        return process_fn(manifest)
    except Exception:
        return []  # Return an empty list on failure

def process_all_manifests(manifests: List[str], process_fn: Callable = extract_url, num_workers: int = None) -> List[str]:
    """Processes all manifests in parallel and aggregates installer URLs."""
    with multiprocess.Pool(num_workers) as pool:
        results = list(tqdm(pool.imap(lambda manifest: process_manifest(manifest, process_fn), manifests), total=len(manifests)))

    return results

urls = [url for sublist in process_all_manifests(manifests, num_workers=8) for url in sublist]
json.dump(urls, open("/Volumes/New Volume/malware-detection-dataset/winget-urls.json", "w"))

In [4]:
def parse_url(url):
    url1 = urlparse(url)
    return url1.netloc, url1.path, url1.params, url1.query, url1.fragment

def url_similarity(url1, url2):
    host1, path1, params1, query1, fragment1 = parse_url(url1)
    host2, path2, params2, query2, fragment2 = parse_url(url2)

    host_sim = jellyfish.levenshtein_distance(host1, host2)
    path_sim = jellyfish.levenshtein_distance(path1, path2)


    return 0.5 * host_sim + 0.5 * path_sim 

In [2]:
N_URLS = 5000

with open("/Volumes/New Volume/malware-detection-dataset/winget-urls.json", "r") as f: 
    content = list(set(json.load(f)))

def compute_distance_matrix(
    strings: List[str],
    max_workers: int = 12,
    chunk_size: int = 1000,
    sim_fn = jellyfish.levenshtein_distance
) -> np.ndarray:
    def process_chunk(pairs):
        results = []
        for i, j in pairs:
            dist = sim_fn(strings[i], strings[j])
            results.append((i, j, dist))
        return results

    n = len(strings)
    distance_matrix = np.zeros((n, n), dtype=np.uint16)

    total_pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
    chunks = [total_pairs[i:i + chunk_size] for i in range(0, len(total_pairs), chunk_size)]

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(process_chunk, chunk) for chunk in chunks]
        for future in tqdm(as_completed(futures), total=len(futures), desc="Computing distances", leave=False):
            for i, j, dist in future.result():
                distance_matrix[i, j] = dist
                distance_matrix[j, i] = dist  

    return distance_matrix

np.random.shuffle(content)
urls_to_compare = content[:N_URLS]
similarity = compute_distance_matrix(urls_to_compare, sim_fn=url_similarity)

FileNotFoundError: [Errno 2] No such file or directory: '/Volumes/New Volume/malware-detection-dataset/winget-urls.json'

In [1]:
def plot_histogram_with_gaussian(data: Sequence[float], bins: int = 30, title: str = "Histogram with Gaussian Overlay"):
    data = np.array(data)
    mu, std = data.mean(), data.std()

    plt.figure(figsize=(8, 5), dpi=300)
    count, bins_edges, _ = plt.hist(data, bins=bins, density=True, alpha=0.6, color='tab:blue', edgecolor='black')

    lower = mu - (3 * std)
    upper = mu + (3 * std)
    x = np.linspace(bins_edges[0], bins_edges[-1], 1000)
    x_within_dist = np.linspace(lower, upper, 1000)
    pdf = norm.pdf(x, mu, std)
    pdf_within_dist = norm.pdf(x_within_dist, mu, std)
    plt.plot(x, pdf, '-', linewidth=2, label=f'N({mu:.2f}, {std:.2f})', c='tab:orange')
    plt.fill_between(x_within_dist, 0, pdf_within_dist, color='tab:orange', alpha=0.5)

    plt.title(title)
    plt.xlabel("Similarity")
    plt.ylabel("Density")
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_histogram_with_gaussian(similarity.flatten(), bins=50, title="Distribution of URL Similarity")

NameError: name 'Sequence' is not defined

In [ ]:
size = similarity.shape[0]
n = 10

# Mask out the lower triangle and diagonal
triu_indices = np.triu_indices(size, k=1)
values = similarity[triu_indices]

# Get the indices of the top-n values
top_n_indices = np.argsort(values)[n:]

# Map back to similarity indices
top_pairs = [(triu_indices[0][i], triu_indices[1][i], values[i]) for i in top_n_indices]
top_pairs

In [16]:
for i, url in enumerate(urls_to_compare[:10]):
    s = similarity[i, i + 1:]
    print(np.where(s < 10))
    

(
    array([   1,   85,   98,  109,  268,  296,  322,  480,  515,  605,  643,
        647,  648,  814,  842,  978, 1065, 1211, 1244, 1262, 1269, 1310,
       1337, 1347, 1355, 1372, 1469, 1554, 1634, 1709, 1847, 1891, 2016,
       2028, 2125, 2135, 2264, 2270, 2321, 2502, 2525, 2552, 2641, 2853,
       2882, 2953, 2960, 3029, 3107, 3250, 3268, 3357, 3395, 3426, 3452,
       3519, 3644, 3704, 3762, 3833, 3847, 3944, 3983, 4006, 4071, 4083,
       4115, 4177, 4217, 4232, 4280, 4336, 4353, 4445, 4645, 4710, 4736,
       4754, 4770, 4797, 4904, 4985, 4987]),
)

(array([], dtype=int64),)

(
    array([  83,   96,  107,  266,  294,  320,  478,  513,  603,  641,  645,
        646,  812,  840,  976, 1063, 1209, 1242, 1260, 1267, 1308, 1335,
       1345, 1353, 1370, 1467, 1552, 1632, 1707, 1845, 1889, 2014, 2026,
       2123, 2133, 2262, 2268, 2319, 2500, 2523, 2550, 2639, 2851, 2880,
       2951, 2958, 3027, 3105, 3248, 3266, 3355, 3393, 3424, 3450, 3517,
       3642, 3702, 3760, 3831, 3845, 3942, 3981, 4004, 4069, 4081, 4113,
       4175, 4215, 4230, 4278, 4334, 4351, 4443, 4643, 4708, 4734, 4752,
       4768, 4795, 4902, 4983, 4985]),
)

(array([], dtype=int64),)

(array([], dtype=int64),)

(
    array([   4,  417, 1090, 1284, 1440, 1512, 1927, 2171, 2472, 2578, 2678,
       2694, 2832, 3294, 3524, 3688, 3691, 4033, 4411, 4783, 4956]),
)

(
    array([  37,   52,   56,  106,  116,  157,  160,  165,  181,  192,  200,
        205,  209,  223,  261,  277,  298,  313,  334,  338,  339,  398,
        401,  410,  422,  441,  446,  461,  479,  480,  482,  483,  492,
        511,  521,  539,  547,  554,  607,  633,  634,  635,  647,  661,
        667,  707,  723,  724,  727,  740,  745,  754,  763,  771,  788,
        790,  796,  840,  849,  854,  855,  874,  883,  903,  912,  920,
        950,  968,  988,  989, 1010, 1061, 1065, 1074, 1077, 1085, 1164,
       1172, 1178, 1181, 1182, 1194, 1195, 1207, 1229, 1240, 1249, 1255,
       1271, 1274, 1286, 1290, 1307, 1315, 1360, 1377, 1387, 1402, 1421,
       1446, 1469, 1480, 1502, 1513, 1532, 1536, 1557, 1558, 1566, 1577,
       1579, 1581, 1582, 1586, 1596, 1604, 1610, 1623, 1651, 1682, 1688,
       1708, 1710, 1726, 1751, 1764, 1776, 1777, 1783, 1799, 1804, 1817,
       1836, 1840, 1874, 1878, 1887, 1892, 1912, 1932, 1943, 1958, 1971,
       1974, 1987, 1994, 2005, 2035, 2038, 2096, 2111, 2112, 2169, 2186,
       2187, 2190, 2196, 2200, 2212, 2216, 2217, 2227, 2245, 2249, 2293,
       2300, 2325, 2337, 2344, 2371, 2380, 2382, 2384, 2398, 2402, 2406,
       2411, 2415, 2423, 2447, 2451, 2452, 2457, 2461, 2483, 2487, 2510,
       2521, 2541, 2542, 2556, 2578, 2579, 2587, 2606, 2608, 2634, 2636,
       2663, 2680, 2691, 2710, 2711, 2720, 2725, 2727, 2754, 2765, 2775,
       2776, 2784, 2788, 2815, 2840, 2844, 2885, 2910, 2924, 2932, 2976,
       2979, 2988, 2991, 3017, 3024, 3026, 3033, 3037, 3051, 3052, 3089,
       3096, 3104, 3114, 3124, 3128, 3137, 3142, 3145, 3148, 3166, 3178,
       3196, 3197, 3200, 3205, 3208, 3224, 3225, 3239, 3248, 3283, 3285,
       3296, 3299, 3303, 3305, 3346, 3373, 3415, 3428, 3429, 3447, 3448,
       3457, 3461, 3466, 3481, 3496, 3498, 3526, 3534, 3540, 3563, 3575,
       3580, 3582, 3588, 3592, 3602, 3611, 3623, 3627, 3634, 3637, 3644,
       3654, 3667, 3677, 3686, 3689, 3709, 3734, 3760, 3767, 3771, 3806,
       3839, 3844, 3857, 3881, 3894, 3933, 3959, 4001, 4002, 4022, 4057,
       4082, 4108, 4121, 4129, 4130, 4158, 4172, 4175, 4179, 4203, 4209,
       4216, 4242, 4243, 4253, 4254, 4272, 4273, 4298, 4303, 4308, 4340,
       4345, 4382, 4401, 4408, 4422, 4431, 4437, 4472, 4478, 4485, 4516,
       4526, 4568, 4583, 4610, 4614, 4649, 4654, 4661, 4662, 4680, 4696,
       4725, 4731, 4737, 4747, 4778, 4780, 4806, 4830, 4851, 4860, 4872,
       4876, 4879, 4899, 4905, 4958, 4965]),
)

(
    array([  43,   74,   92,   93,  101,  136,  145,  196,  212,  214,  245,
        246,  251,  346,  388,  405,  407,  458,  465,  468,  570,  584,
        599,  625,  643,  678,  689,  697,  699,  707,  713,  736,  784,
        794,  799,  830,  861,  878,  939,  959, 1083, 1101, 1106, 1110,
       1112, 1121, 1130, 1166, 1172, 1196, 1235, 1295, 1312, 1317, 1346,
       1406, 1423, 1488, 1520, 1522, 1549, 1611, 1614, 1633, 1656, 1674,
       1680, 1683, 1710, 1790, 1810, 1824, 1836, 1847, 1848, 1864, 1867,
       1885, 1949, 1951, 1955, 1966, 2005, 2053, 2054, 2060, 2068, 2103,
       2231, 2261, 2284, 2300, 2389, 2461, 2478, 2481, 2497, 2533, 2572,
       2637, 2652, 2656, 2659, 2703, 2737, 2738, 2743, 2790, 2794, 2824,
       2883, 2918, 2944, 2967, 3044, 3083, 3084, 3086, 3094, 3106, 3120,
       3126, 3167, 3242, 3248, 3264, 3290, 3327, 3349, 3380, 3391, 3410,
       3444, 3458, 3469, 3544, 3559, 3569, 3605, 3608, 3662, 3665, 3703,
       3711, 3731, 3740, 3786, 3792, 3809, 3834, 3839, 3847, 3863, 3910,
       3959, 3967, 3972, 3975, 3990, 3996, 4007, 4012, 4036, 4077, 4095,
       4109, 4116, 4136, 4139, 4143, 4154, 4167, 4175, 4187, 4188, 4198,
       4237, 4249, 4259, 4260, 4267, 4303, 4351, 4357, 4367, 4423, 4429,
       4439, 4454, 4469, 4488, 4495, 4614, 4672, 4699, 4712, 4722, 4735,
       4773, 4789, 4800, 4832, 4834, 4853, 4870, 4896, 4956, 4962]),
)

(array([1363, 4407, 4431, 4616]),)

(array([289]),)